In [1]:
"""
Module imports
"""

import json
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer

In [3]:
"""
Directory configuration
"""
def _project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "db").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find project root from {start}")

ROOT = _project_root()
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
"""
Parameter configuration
"""

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 140

N_WAY = 6
N_SHOT = 10
N_QUERY = 15

EPOCHS = 10
META_TRAIN_EPISODES = 500
META_TEST_EPISODES = 150
QUERY_EPISODES = 300
LEARNING_RATE = 2e-5

EARLY_STOPPING_PATIENCE = 2
COSINE_TEMPERATURE = 0.1

GPU_COUNT = torch.cuda.device_count()
DEVICE = torch.device("cuda:0" if GPU_COUNT else "cpu")
USE_DATA_PARALLEL = GPU_COUNT > 1

DATA_PATHS = {
    "goemotions_train": ROOT / "db" / "goemotions" / "train.tsv",
    "goemotions_dev": ROOT / "db" / "goemotions" / "dev.tsv",
    "goemotions_test": ROOT / "db" / "goemotions" / "test.tsv",
    "goemotions_emotions": ROOT / "db" / "goemotions" / "emotions.txt",
    "goemotions_ekman": ROOT / "db" / "goemotions" / "ekman_mapping.json",
}

In [ ]:
"""
Seed setting for reproducibility
"""

RANDOM_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
print("Using device:", DEVICE)
print("GPUs available:", GPU_COUNT)
print("Data parallelism:", "enabled" if USE_DATA_PARALLEL else "disabled")


In [ ]:
"""
Label normalization
"""

CATEGORY_LABELS = ["anger", "disgust", "joy", "sadness", "surprise", "neutral"]
CATEGORY_TO_ID = {label: index for index, label in enumerate(CATEGORY_LABELS)}
EXCLUDED_CATEGORIES = {"fear"}

with open(DATA_PATHS["goemotions_emotions"], encoding="utf-8") as emotions_file:
    GOEMOTIONS_LABELS = [line.strip() for line in emotions_file if line.strip()]

with open(DATA_PATHS["goemotions_ekman"], encoding="utf-8") as ekman_file:
    EKMAN_MAPPING = json.load(ekman_file)

GOEMOTIONS_ID_TO_CATEGORY = {}
for category, emotions in EKMAN_MAPPING.items():
    if category in CATEGORY_TO_ID:
        for emotion in emotions:
            GOEMOTIONS_ID_TO_CATEGORY[GOEMOTIONS_LABELS.index(emotion)] = category
GOEMOTIONS_ID_TO_CATEGORY[GOEMOTIONS_LABELS.index("neutral")] = "neutral"

CMED_FIRST_CHARACTER_TO_CATEGORY = {24179: "neutral", 38283: "joy", 24754: "sadness", 24929: "anger", 39514: "surprise", 24656: "fear", 21388: "disgust", 30097: "surprise"}

def _first_goemotions_category(value):
    emotion_id = int(str(value).split(",")[0])
    return GOEMOTIONS_ID_TO_CATEGORY.get(emotion_id)

def _normalize_cmed_label(value):
    label = str(value).strip()
    if label in CATEGORY_TO_ID or label == "fear":
        return label
    return CMED_FIRST_CHARACTER_TO_CATEGORY.get(ord(label[0])) if label else None

def _filter_and_encode(dataframe):
    frame = dataframe.dropna(subset=["text", "category"]).copy()
    frame["category"] = frame["category"].astype(str).str.strip().str.lower()
    frame = frame[~frame["category"].isin(EXCLUDED_CATEGORIES)]
    frame = frame[frame["category"].isin(CATEGORY_TO_ID)].copy()
    frame["label"] = frame["category"].map(CATEGORY_TO_ID).astype(int)
    return frame[["text", "category", "label"]].reset_index(drop=True)

print("Categories:", CATEGORY_TO_ID)
print("Excluded categories:", sorted(EXCLUDED_CATEGORIES))


Categories: {'anger': 0, 'disgust': 1, 'joy': 2, 'sadness': 3, 'surprise': 4, 'neutral': 5}
Excluded categories: ['fear']


In [ ]:
"""
Load GoEmotions, CMED, and MYBully
"""

def load_goemotions(paths):
    frames = []
    for split_name in ("train", "dev", "test"):
        frame = pd.read_csv(paths[f"goemotions_{split_name}"], sep="\t", header=None, names=["text", "emotion", "userid"])
        frame = frame.dropna(subset=["text", "emotion"]).copy()
        fear_id = str(GOEMOTIONS_LABELS.index("fear"))
        frame = frame[~frame["emotion"].astype(str).str.split(",").apply(lambda ids: fear_id in ids)].copy()
        frame["category"] = frame["emotion"].apply(_first_goemotions_category)
        frames.append(_filter_and_encode(frame))
    return pd.concat(frames, ignore_index=True)

def load_cmed():
    path = "hf://datasets/Johnson8187/Chinese_Multi-Emotion_Dialogue_Dataset/data.csv"
    frame = pd.read_csv(path, usecols=["text", "emotion"])
    frame["category"] = frame["emotion"].apply(_normalize_cmed_label)
    return _filter_and_encode(frame)

def load_mybully():
    path = "hf://datasets/mohanrj/MYBully/"
    train = pd.read_csv(path + "train.csv", usecols=["Tweet", "Emotion"])
    valid = pd.read_csv(path + "val.csv", usecols=["Tweet", "Emotion"])
    train = train.rename(columns={"Tweet": "text", "Emotion": "category"})
    valid = valid.rename(columns={"Tweet": "text", "Emotion": "category"})
    train["category"] = train["category"].astype(str).str.strip().str.lower().replace({"happiness": "joy"})
    valid["category"] = valid["category"].astype(str).str.strip().str.lower().replace({"happiness": "joy"})
    return _filter_and_encode(train), _filter_and_encode(valid)

goemotions_df = load_goemotions(DATA_PATHS)
cmed_df = load_cmed()
mybully_meta_test_df, mybully_query_df = load_mybully()
meta_train_df = pd.concat([goemotions_df, cmed_df], ignore_index=True)

print("Meta-train:", len(meta_train_df))
print("MYBully meta-test:", len(mybully_meta_test_df))
print("MYBully query/validation:", len(mybully_query_df))

Meta-train: 56035
MYBully meta-test: 3449
MYBully query/validation: 1152
Meta-train labels:
 category
joy         21721
neutral     16726
anger        6700
surprise     6453
sadness      3710
disgust       725
Name: count, dtype: int64
MYBully query labels:
 category
anger       620
neutral     240
joy         113
disgust      83
sadness      69
surprise     27
Name: count, dtype: int64


In [ ]:
print("Meta-train labels:\n", meta_train_df["category"].value_counts())

In [ ]:
print("MYBully query labels:\n", mybully_query_df["category"].value_counts())

In [ ]:
"""
Prototype-network episode sampling
"""

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, timeout=120)

def _sample_episode(frame, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY):
    available_labels = sorted(frame["label"].unique())
    if len(available_labels) < n_way:
        raise ValueError(f"Need {n_way} categories, found {len(available_labels)}")
    episode_labels = random.sample(available_labels, n_way)
    support_texts, support_labels = [], []
    query_texts, query_labels = [], []
    for local_label, global_label in enumerate(episode_labels):
        rows = frame[frame["label"] == global_label]
        required = n_shot + n_query
        if len(rows) < required:
            raise ValueError(f"Category {global_label} has {len(rows)} rows; need {required}")
        selected = rows.sample(required, replace=False)
        support_texts.extend(selected.iloc[:n_shot]["text"].tolist())
        support_labels.extend([local_label] * n_shot)
        query_texts.extend(selected.iloc[n_shot:]["text"].tolist())
        query_labels.extend([local_label] * n_query)
    return support_texts, support_labels, query_texts, query_labels, episode_labels

def _tokenize(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")

def _episode_tensors(frame):
    support_texts, support_labels, query_texts, query_labels, episode_labels = _sample_episode(frame)
    return (_tokenize(support_texts), torch.tensor(support_labels, device=DEVICE),
            _tokenize(query_texts), torch.tensor(query_labels, device=DEVICE), episode_labels)

def _create_episode_set(frame, number_of_episodes):
    return [_episode_tensors(frame) for _ in range(number_of_episodes)]


In [ ]:
"""
Prototypical Network and training
"""

class PrototypeNetwork(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        if USE_DATA_PARALLEL:
            self.encoder = nn.DataParallel(self.encoder)

    def embed(self, batch):
        outputs = self.encoder(input_ids=batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE))
        mask = batch["attention_mask"].to(DEVICE).unsqueeze(-1)
        pooled = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        return F.normalize(pooled, p=2, dim=1)

    def episode_loss(self, support_batch, support_labels, query_batch, query_labels):
        support_embeddings = self.embed(support_batch)
        query_embeddings = self.embed(query_batch)
        prototypes = torch.stack([support_embeddings[support_labels == label].mean(dim=0) for label in range(N_WAY)])
        prototypes = F.normalize(prototypes, p=2, dim=1)
        logits = torch.mm(query_embeddings, prototypes.transpose(0, 1)) / COSINE_TEMPERATURE
        return F.cross_entropy(logits, query_labels), logits.argmax(dim=1)

model = PrototypeNetwork(MODEL_NAME).to(DEVICE)
print("Prototype encoder device:", DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
history = []
best_validation_accuracy = float("-inf")
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses, correct, total = [], 0, 0
    train_episodes = _create_episode_set(meta_train_df, META_TRAIN_EPISODES)

    for support_batch, support_labels, query_batch, query_labels, _ in train_episodes:
        optimizer.zero_grad()
        loss, predictions = model.episode_loss(
            support_batch, support_labels, query_batch, query_labels
        )
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

        correct += int((predictions == query_labels).sum())
        total += len(query_labels)

    validation_episodes = _create_episode_set(mybully_query_df, QUERY_EPISODES)
    model.eval()
    validation_correct, validation_total = 0, 0

    with torch.no_grad():
        for support_batch, support_labels, query_batch, query_labels, _ in validation_episodes:
            _, predictions = model.episode_loss(
                support_batch, support_labels, query_batch, query_labels
            )
            validation_correct += int((predictions == query_labels).sum())
            validation_total += len(query_labels)

    validation_accuracy = validation_correct / validation_total
    metrics = {
        "epoch": epoch,
        "loss": float(np.mean(losses)),
        "accuracy": correct / total,
        "validation_accuracy": validation_accuracy
    }
    history.append(metrics)

    print(f"Epoch {epoch:02d}/{EPOCHS} - loss: {metrics['loss']:.4f} - accuracy: {metrics['accuracy']:.4f} - validation accuracy: {validation_accuracy:.4f}")

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(
            {
                "model_state_dict": best_model_state,
                "history": history,
                "best_epoch": best_epoch,
                "best_validation_accuracy": best_validation_accuracy
            },
            MODELS_DIR / "metalearning_prototypical_network_best.pt"
        )
        print(f"Saved best model at epoch {best_epoch}")
        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}; best validation accuracy: {best_validation_accuracy:.4f}")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "history": history,
        "best_epoch": best_epoch,
        "best_validation_accuracy": best_validation_accuracy
    }, 
    MODELS_DIR / "metalearning_prototypical_network.pt"
)


In [ ]:
"""
MYBully meta-test and query/validation evaluation
"""

def evaluate_episodes(frame, number_of_episodes, episodes=None):
    model.eval()
    all_targets, all_predictions = [], []
    episodes = episodes or _create_episode_set(frame, number_of_episodes)

    with torch.no_grad():
        for support_batch, support_labels, query_batch, query_labels, episode_labels in episodes:
            _, predictions = model.episode_loss(
                support_batch, support_labels, query_batch, query_labels
            )
            all_targets.extend(
                [episode_labels[label] for label in query_labels.cpu().tolist()]
            )
            all_predictions.extend(
                [episode_labels[label] for label in predictions.cpu().tolist()]
            )

    labels = list(range(N_WAY))
    return all_targets, all_predictions

meta_test_targets, meta_test_predictions = evaluate_episodes(mybully_meta_test_df, META_TEST_EPISODES)
query_targets, query_predictions = evaluate_episodes(mybully_query_df, QUERY_EPISODES)

print("MYBully meta-test report")
print(classification_report(meta_test_targets, meta_test_predictions, labels=list(range(N_WAY)), target_names=CATEGORY_LABELS, zero_division=0))

In [ ]:
print("MYBully query/validation report")
print(classification_report(query_targets, query_predictions, labels=list(range(N_WAY)), target_names=CATEGORY_LABELS, zero_division=0))

In [ ]:
print("Meta-test confusion matrix:\n", confusion_matrix(meta_test_targets, meta_test_predictions, labels=list(range(N_WAY))))

In [ ]:
print("Query/validation confusion matrix:\n", confusion_matrix(query_targets, query_predictions, labels=list(range(N_WAY))))